# GDELT Data Inspection

This notebook helps you inspect each column (dtype, nulls, uniqueness, example values) so you can decide what to drop in future queries.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 120)

project_root = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
parquet_path = project_root / 'data' / 'processed' / 'ukraine_russia_war.parquet'

if not parquet_path.exists():
    raise FileNotFoundError(f'Parquet file not found: {parquet_path}')

df = pd.read_parquet(parquet_path)
print(f'Loaded: {parquet_path}')
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')

In [ ]:
df.head(10)

In [ ]:
def short_examples(series: pd.Series, n: int = 3):
    vals = series.dropna().astype(str).drop_duplicates().head(n).tolist()
    return [v[:100] + ('…' if len(v) > 100 else '') for v in vals]

rows = []
n_rows = len(df)

for col in df.columns:
    s = df[col]
    null_count = int(s.isna().sum())
    non_null_count = n_rows - null_count
    unique_count = int(s.nunique(dropna=True))

    if pd.api.types.is_string_dtype(s) or s.dtype == 'object':
        avg_len = float(s.dropna().astype(str).str.len().mean()) if non_null_count > 0 else np.nan
    else:
        avg_len = np.nan

    rows.append({
        'column': col,
        'dtype': str(s.dtype),
        'null_count': null_count,
        'null_%': round((null_count / n_rows) * 100, 2),
        'non_null_%': round((non_null_count / n_rows) * 100, 2),
        'unique_count': unique_count,
        'unique_%_of_non_null': round((unique_count / non_null_count) * 100, 2) if non_null_count else np.nan,
        'avg_string_length': round(avg_len, 2) if not np.isnan(avg_len) else np.nan,
        'example_values': short_examples(s, n=3),
    })

profile = pd.DataFrame(rows).sort_values(['null_%', 'unique_count'], ascending=[False, False])
profile

In [ ]:
# Quick candidates for dropping (heuristics only)
high_null_threshold = 95.0

constant_cols = profile.loc[profile['unique_count'] <= 1, 'column'].tolist()
mostly_null_cols = profile.loc[profile['null_%'] >= high_null_threshold, 'column'].tolist()

print('Potential drop candidates (review before dropping):')
print(f'- Constant columns (<=1 unique non-null value): {constant_cols}')
print(f'- Mostly null columns (>= {high_null_threshold}% null): {mostly_null_cols}')

In [ ]:
def inspect_column(col: str, top_n: int = 20):
    if col not in df.columns:
        raise ValueError(f'Column not found: {col}')

    s = df[col]
    print(f'Column: {col}')
    print(f'Dtype: {s.dtype}')
    print(f'Nulls: {s.isna().sum():,} / {len(s):,} ({(s.isna().mean()*100):.2f}%)')
    print(f'Unique (non-null): {s.nunique(dropna=True):,}')
    print('\nTop values:')
    display(s.astype(str).value_counts(dropna=False).head(top_n).to_frame('count'))

# Example:
inspect_column('SourceCommonName', top_n=15)

In [ ]:
msn_articles = df[df['SourceCommonName'].astype(str).str.lower() == 'msn.com'].copy()

print(f"Found {len(msn_articles):,} articles from msn.com")

cols_to_show = [c for c in ['DATE', 'SourceCommonName', 'DocumentIdentifier'] if c in msn_articles.columns]
if cols_to_show:
    display(msn_articles[cols_to_show].head(100))
else:
    display(msn_articles.head(100))